# UQ-Edge: Does Quantization Break Model Confidence?

**Systematic study of calibration in quantized small language models.**

- 8 models (135M → 3B) × 3 quant levels (FP16, INT8, NF4) × 5 benchmarks
- Phase 1: Inference + logit capture (GPU) → saves `.npz` files
- Phase 2: Metrics computation (CPU) → ECE, AUROC, Brier
- Phase 3: Self-consistency (GPU, optional)
- Phase 4: Generate paper figures

**Runtime**: ~4-6 hours total on A100 for full Phase 1.

---

## 0. Setup

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Create project directory on Drive
!mkdir -p /content/drive/MyDrive/uq-edge
!mkdir -p /content/drive/MyDrive/uq-edge/raw_outputs
!mkdir -p /content/drive/MyDrive/uq-edge/results
!mkdir -p /content/drive/MyDrive/uq-edge/plots
print("Drive mounted and directories created.")

In [ ]:
import urllib.request
import os

# Token is stored in Colab Secrets (sidebar: key icon > add "GH_TOKEN")
# This keeps it out of the notebook file so GitHub won't block your push.
from google.colab import userdata
TOKEN = userdata.get("GH_TOKEN")

REPO = "AniruddhaChattopadhyay/Research-Battery-Lora"
BRANCH = "main"
PROJECT_DIR = "/content/drive/MyDrive/uq-edge"

FILES = [
    "config.py",
    "model_utils.py",
    "data_utils.py",
    "inference.py",
    "uq_methods.py",
    "metrics.py",
    "plotting.py",
    "run_all.py",
    "quick_test.py",
]

for f in FILES:
    url = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}/uq-edge/{f}"
    req = urllib.request.Request(url, headers={"Authorization": f"token {TOKEN}"})
    content = urllib.request.urlopen(req).read()
    with open(f"{PROJECT_DIR}/{f}", "wb") as out:
        out.write(content)
    print(f"  Updated {f} ({len(content):,} bytes)")

print(f"\nAll {len(FILES)} files synced to {PROJECT_DIR}")

# Verify
py_files = [f for f in os.listdir(PROJECT_DIR) if f.endswith('.py')]
print(f"Python files present: {sorted(py_files)}")

In [ ]:
# Install dependencies
# Colab already has torch, transformers, etc. — just ensure versions + extras
!pip install -q bitsandbytes accelerate datasets scikit-learn seaborn scipy pandas

# Verify key imports
import torch, transformers, bitsandbytes
print(f"torch={torch.__version__}, transformers={transformers.__version__}, bnb={bitsandbytes.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Set working directory and add to Python path
import sys
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Quick sanity test — runs 10 samples, verifies full pipeline
!python quick_test.py

## 1. Phase 1 — Inference (GPU Required)

Each cell runs **one model at all quant levels and all benchmarks**. This means 3 quant × 5 bench = 15 runs per model.

**Resume-safe**: If Colab disconnects, just re-run the cell. It skips already-completed `.npz` files.

Estimated time per model: **20-40 min** (depends on model size).

In [ ]:
# Model 1: SmolLM2-135M (~20 min)
!python run_all.py --phase1 --model SmolLM2-135M

In [ ]:
# Model 2: SmolLM2-1.7B (~25 min)
!python run_all.py --phase1 --model SmolLM2-1.7B

In [ ]:
# Model 3: Qwen2.5-0.5B (~20 min)
!python run_all.py --phase1 --model Qwen2.5-0.5B

In [ ]:
# Model 4: Qwen2.5-1.5B (~25 min)
!python run_all.py --phase1 --model Qwen2.5-1.5B

In [ ]:
# Model 5: Qwen2.5-3B (~30 min)
!python run_all.py --phase1 --model Qwen2.5-3B

In [ ]:
# Model 6: TinyLlama-1.1B (~25 min)
!python run_all.py --phase1 --model TinyLlama-1.1B

In [ ]:
# Model 7: Llama-3.2-1B (~25 min)
# Note: may need HF token for gated model. Uncomment below if needed:
# from huggingface_hub import login
# login(token="hf_YOUR_TOKEN_HERE")
!python run_all.py --phase1 --model Llama-3.2-1B

In [ ]:
# Model 8: Llama-3.2-3B (~40 min)
!python run_all.py --phase1 --model Llama-3.2-3B

In [ ]:
# Check Phase 1 progress — how many .npz files do we have?
import os
npz_files = [f for f in os.listdir("raw_outputs") if f.endswith(".npz")]
print(f"Completed: {len(npz_files)} / 120 expected (8 models × 3 quants × 5 benchmarks)")
print("\nFiles:")
for f in sorted(npz_files):
    size = os.path.getsize(f"raw_outputs/{f}") / 1024
    print(f"  {f} ({size:.0f} KB)")

## 2. Phase 2 — Compute Metrics (CPU Only)

This is fast (~1 min). Reads `.npz` files, computes ECE/AUROC/Brier for each UQ method, saves JSON.

In [ ]:
!python run_all.py --phase2

## 3. Results Summary

In [ ]:
!python run_all.py --summary

## 4. Generate Paper Figures

In [ ]:
!python run_all.py --plots

In [ ]:
# Display key figures inline
from IPython.display import Image, display
import os

plot_dir = "plots"
key_plots = [
    "ece_heatmap_msp.png",
    "ece_delta_msp.png",
    "reliability_msp.png",
    "auroc_comparison.png",
    "ece_heatmap_temp_scaled_msp.png",
]

for fname in key_plots:
    path = os.path.join(plot_dir, fname)
    if os.path.exists(path):
        print(f"\n{'='*60}")
        print(f"  {fname}")
        print(f"{'='*60}")
        display(Image(filename=path, width=800))

## 5. Phase 3 — Self-Consistency (Optional, GPU Required)

Generates k=5 sampled responses per question to measure agreement-based confidence.
Only needed if we want to compare sampling-based UQ against single-pass methods.

**Runtime**: ~2-3 hours additional. Run only after reviewing Phase 2 results.

In [ ]:
# Uncomment to run Phase 3
# !python run_all.py --phase3

# Then recompute metrics to include self-consistency scores
# !python run_all.py --phase2 --plots --summary

## 6. Download Results

In [ ]:
# Zip results for download (raw_outputs are already on Drive)
!cd /content/drive/MyDrive/uq-edge && zip -r /content/uq_edge_results.zip results/ plots/

# Download the zip
from google.colab import files
files.download('/content/uq_edge_results.zip')
print("Download started. Check your browser downloads.")

## Quick Reference

| What | Command |
|------|---------|
| Run everything for one model | `!python run_all.py --phase1 --model SmolLM2-135M` |
| Recompute metrics | `!python run_all.py --phase2` |
| Regenerate plots | `!python run_all.py --plots` |
| Print summary table | `!python run_all.py --summary` |
| Force re-run (ignore existing) | Add `--force` to any command |
| Run single quant level | `!python run_all.py --phase1 --model SmolLM2-135M --quant nf4` |